In [ ]:
day08

0. 복습

시간, 날짜 데이터 처리

1) 문자열 -> 날짜 타입 : pd.to_datetime()
2) .dt로 날짜 뽑아내기
	ex) dt.yeqr(연도)
3) 날짜 - 날짜 => 기간(Timedelta)
 ex) 407days
4) 날짜 <= 날짜 : 특정 날짜 범위 필터링

기초통계(기술통계) 
 : 데이터를 몇개의 대표 숫자로 요약

1) 대표값(평균, 중앙값, 최빈값)

2) 산포도(표준편차, 분산, 분위수)

상관관계
 : 두 값이 함께 움직이는 정도
* 상관계수 : -1 ~ +1

1. 범주형 인코딩 
 : 문자열로된 범주형 데이터를, 머신러닝이 이해할 수 있는 숫자로
   바꾸는 것
cf) 범주형 데이터 : 카테고리 데이터
	ex) 혈액형, MBTI
- 머신러닝 모델은 내부적으로 숫자 연산을 중심으로 한다.
  그래서 'male'같은 문자열은 연산을 할 수 없다
- 범주형 데이터를 숫자로 바꿔야 연산을 할 수 있다
- 범주형 인코딩 방법은 2가지가 있다
	(범주에 순서가 있느냐 없느냐에 따라 구분된다)
	라벨 인코딩 : 각 범주에 번호를 매긴다
		ex) small ->, medium -> 2, :arge -> 3
	원-핫 인코딩 : 범주마다 칸(열)을 만들어 True/False로
		       표현한다

1) 라벨 인코딩 : 범주에 번호 매기기(순서형)
 : 각 범주에 정수 번호를 매기는 방법
- 순서가 있는 범주(순서형)에 적합하다.
- 학점, 학년, 만족도 처럼 크기 비교가 자연스러운 값
 => 순서형 범주 데이터
- astype("category").cat.codes : 자동으로 번호 매기기
		(범주형 데이터를 사전순으로 정렬해 0부터 매긴다)

2) 원-핫 인코딩 : 범주마다 열 만들기(명목형)
 : 범주 하나하나를 각각의 열로 만들고, 해당하면 True, 아니면 False
- 순서가 없는 범주(명목형)에 적합하다
- 성별, 혈액형, MBTI 처럼 값 사이에 크기 및 순서가 없는 것들
- pd.get_dummies() 사용

2. EDA(탐색적 데이터 분석)
 : 데이터를 본격적으로 사용하기 전에, 전처리 하면서
   그 안에 무엇이 있는지 살펴보는 전 과정

* EDA 순서
- EDA 탐색적 데이터 분석은 대략 아래 순서로 진행된다
(1) 불러오기 & 첫 탐색 - 크기, 자료형, 결측치 훑어보기
(2) 결측치 처리 - 제거 혹은 삭제
(3) 이상치 점검 - 확인 혹은 처리
(4) 파생 변수 만들기 - 기존 열로 새 특성 생성
(5) 인코딩 & 스케일링
(6) 상관, 시각화 확인

In [ ]:
## 범주형 인코딩
import pandas as pd
import seaborn as sns

# 타이타닉 데이터
titanic = sns.load_dataset("titanic")
titanic.head()
print(titanic['class'].value_counts())
### 라벨 인코딩
# map을 사용하여 직접 번호 지정하기(라벨 붙이기)
# {범주:번호} 사용하여 번호 매기기
titanic['class_num'] = titanic['class'].map({"First":1, "Second":2, "Third":3})
titanic[['class', 'class_num']].head()
# astype("category") : 카테고리 타입으로 형변환 한뒤
# .cat.codes 로 자동 번호 부여(First=0, Second=1, Third=2)
print(titanic['class'].astype('category').cat.codes.head())
### 원-핫 인코딩
# 성별을 원-핫 인코딩
pd.get_dummies(titanic['sex']).head()
# 한 열이 female, male 두 열로 나뉘고, 자기 성별 칸만 True가 된다
# (0번은 male=True, 1번은 female=True)
print(titanic['embarked'].value_counts())

# embarked를 원-핫 인코딩
pd.get_dummies(titanic['embarked']).head()
# 원-핫 인코딩은 열이 늘어나는게 단점이다.
# 사실 범주가 N개면 열은 N-1개면 충분하다
# ex) 성별은 male열 하나만 봐도(True면 남, False면 여)다 알 수 있다
# => drop_first= True 을 사용해 첫 범주 열을 지워 열을 하나 줄인다
pd.get_dummies(titanic['embarked'], drop_first=True).head()
# C열은 사라져서 나옴
# => 이렇게 열을 줄이면 중복(다중공선성)을 막을 수 있다0
# +) get_dummies에 데이터 프레임과 columns=를 주면,
# 지정한 여러 범주 열을 한번에 원-핫 인코딩하고
# 나머지 열은 그대로 둔다

# age는 그대로 두고 + 성별, 승선도시 만 원-핫 인코딩(실제 많이 사용하는 형태)
sub = titanic[['age', 'sex', 'embarked']]
encoded = pd.get_dummies(sub, columns=['sex', 'embarked'])
encoded
# 열 이름이 원래열이름_범주값(embarked_C) 형태로 자동 생성된다
# 앞의 age열은 건드리지 않음
# <범주형 인코딩 실습>
import pandas as pd
import seaborn as sns

# 펭귄 데이터 불러오기
penguins = sns.load_dataset("penguins")
penguins.head()
# 1) 성별 라벨 인코딩 - map으로 Female=>0, Male -> 1
# 로 바꿔서 gender_num열 만들기

penguins['gender_num'] = penguins['sex'].map({"Female" : 0, "Male" : 1})
penguins[['sex', 'gender_num']].head()
# 2) species를 원-핫 인코딩하고
print(penguins['species'].value_counts())

pd.get_dummies(penguins['species'], dtype=int).head()
# 3) species, island, sex 세 범주 열을 한번에 원-핫 인코딩하고
#    drop_first=True를 주고 확인
encoded = pd.get_dummies(
    penguins[['species', 'island', 'sex']],
    drop_first= True
)
print(penguins['island'].value_counts())
encoded.head()
## EDA 탐색적 데이터 분석
### 0. 첫 탐색 - 데이터를 받으면 제일 먼저
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic' # 시각화 한글깨짐 방지

# 타이타닉 불러오기
titanic = sns.load_dataset("titanic")
titanic.head()
# 1) 데이터 크기 확인
print(f"크기(행, 열) : {titanic.shape}")
# 891개 행, 15개의 형
# 2) 열마다의 데이터 타입 확인
print("\n=== 데이터 타입 ===")
print(titanic.dtypes)

# 3) 결측치 확인
print("\n=== 결측치 확인 ===")
print(titanic.isnull().sum())
titanic.info() # 대략적인 데이터 확인
### 1. 결측치 처리
> 거의 다 빈 열은 삭제, 숫자는 중앙값, 문자는 최빈값
# deck : 688개나 비어 있음 -> 열 통째로 삭제
# titanic = titanic.drop(columns='deck')

# age(숫자) : 중앙값으로 채우기(이상치에 강한 대표값)
titanic['age'] = titanic['age'].fillna(titanic['age'].median())

# embarked, embarked_town(문자) : 최빈값으로 채우기
titanic['embarked'] = titanic['embarked'].fillna(
    titanic['embarked'].mode()[0]
)
titanic['embark_town'] = titanic['embark_town'].fillna(
    titanic['embark_town'].mode()[0]
)

print("=== 결측치 확인 ===")
print(titanic.isnull().sum())
# 결측치 처리 완료
### 2. 이상치 점검
- fare(요금)는 대부분 저렴한데 일부가 512까지 치솟는 이상치가 있다
- IQR로 경계를 구해 clip으로 눌러담기
titanic.describe()
# IQR로 방법으로 위 경계 계산
# IQR(사분위 범위) = Q3 - Q1
# 위 경계 = Q3 + 1.5 x IQR
# 아래 경계 = Q1 - 1.5 x IQR

# Q3, Q1 찾기
q1 = titanic['fare'].quantile(0.25)
q3 = titanic['fare'].quantile(0.75)
iqr = q3 - q1 # IQR 계산
# 위쪽 경계값 계산
high = q3 + 1.5 * iqr
print(f"위쪽 경계값 : {high}")

# 위 경계보다 큰 요금을 경계값으로 눌러담기
print(f"clip전 최대값 : {titanic['fare'].max()}") # 512
titanic['fare'] = titanic['fare'].clip(upper=high)
print(f"clip 후 최대값 : {titanic['fare'].max()}")
# 512까지 있던 이상치를 65.63까지 누름(행은 그대로 유지)
### 3. 파생 변수 만들기
- 기존 열을 조합, 변형해 새로운 특성을 만든다
- 가족수 = 형제/배우자 + 부모/자식 + 본인
- 연령대 = 나이를 구간으로 나누기
# 여러 열을 더해 새 열(가족수)
titanic['family_size'] = titanic['sibsp'] + titanic['parch'] + 1

# 나이를 연령대로 구간분할(cut)
titanic['age_group'] = pd.cut(
    titanic['age'],
    bins = [0, 18, 35, 60, 120],
    labels=['어린이', '청년', '중년', '노년']
)
print(titanic['age_group'].value_counts().sort_index())

titanic.head()
### 4. 인코딩 & 스케일링 - 모델 입력 준비
- 문자열 -> 숫자로(인코딩)
- 숫자는 크기를 맞추 스케일링
# 명목형 범주를 원-핫 인코딩
encoded = pd.get_dummies(
    titanic[['sex', 'embarked']],
    columns=['sex', 'embarked']
)
encoded.head()
from sklearn.preprocessing import StandardScaler
# 스케일링 - 범위가 숫자열 (age, fare)을 표준화(평균 0, 표준편차 1)

scaled = pd.DataFrame(
    StandardScaler().fit_transform(titanic[['age', 'fare']]),
    columns= ['age', 'fare']
)
print(scaled.describe())

scaled.head()
### 5. 상관관계로 인사이트 찾기
- 무엇이 생존과 관련 있는지 확인
# 주요 숫자 열 가져오기
cols = ['survived', 'age', 'fare', 'family_size', 'pclass']
titanic[cols].corr(numeric_only=True).round(2)
# 생존에 관련된 속성은 크게 없다
# 그나마, pclass(-0.34)와 fare(0.32)가 그나마 아주 조금 관련이 있다
# => 객실 등급이 높고 요금이 비쌀수록 생존에 아주 미미한 영향이 있을 수 있다
# 상관행렬을 히트맵으로 시각화
plt.rcParams['axes.unicode_minus'] = False # 기호 깨짐 방지

plt.figure(figsize=(6, 5))
sns.heatmap(titanic[cols].corr(numeric_only=True),
           annot=True, cmap='coolwarm', vmin=-1, vmax=1,
           fmt=".2f")
plt.title("타이타닉 상관관계 히트맵")
plt.show()
# <EDA 실습 문제>
import pandas as pd
import seaborn as sns

penguins = sns.load_dataset("penguins")
penguins.head()
# 수치열 : bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g
# 범주열 : species, island, sex

# 1) 열별 결측치 확인
#    수치열은 각 열의 중앙값
#    sex열은 최빈값으로 채우기

# 결측치 확인
print(penguins.isnull().sum())

num_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']

for col in num_cols :
    # 수치열의 결측치를 중앙값으로 대체
    penguins[col] = penguins[col].fillna(penguins[col].median())
# 문자는 최빈값
penguins['sex'] = penguins['sex'].fillna(
    penguins['sex'].mode()[0]
)
print("\n---- 남은 결측치 ----")
print(penguins.isnull().sum())
# 2) 수치열 4개의 상관행렬을 구해, 가장 강한 양의 상관관계를 가지는 열 확인
penguins[num_cols].corr(numeric_only=True).round(2)
# flipper_length_mm 와 body_mass_g 0.87로 강한 양의 상관관계를 가진다
# 날개가 길수록 무겁다

### 과제

## 프로젝트2 day08 과제 — 범주형 인코딩 & EDA 종합실습

### [1부] 범주형 인코딩

seaborn `tips` 데이터로 인코딩을 연습합니다. 아래 셀을 먼저 실행하세요.

- 범주 열 : `sex`(성별), `smoker`(흡연 No/Yes), `day`(요일), `time`(Lunch/Dinner)
- 머신러닝은 숫자만 받으므로, 이런 글자 범주를 **숫자로 인코딩**합니다.

In [ ]:
import pandas as pd
import seaborn as sns

tips = sns.load_dataset("tips")
print(tips.head())

문제 1) 라벨 인코딩 — map으로 번호 매기기

- `smoker`(No/Yes)를 `map`으로 **`No→0`, `Yes→1`** 로 바꿔 새 열 `smoker_num` 에 저장하세요.
- `smoker`·`smoker_num` 을 **앞 5줄** 출력하고, `smoker_num`의 개수도 세어 출력하세요.

<출력결과>

smoker  smoker_num
0     No           0
1     No           0
2     No           0
3     No           0
4     No           0
smoker_num
0    151
1     93
Name: count, dtype: int64

In [ ]:
tips['smoker_num'] = tips['smoker'].map({"No":0, "Yes":1})
print(tips[['smoker', 'smoker_num']].head())
print(tips['smoker_num'].value_counts())

문제 2) 원-핫 인코딩 — get_dummies

- `day`(요일, 4범주)를 **원-핫 인코딩**해 앞 5줄 출력하세요. (힌트 : `pd.get_dummies(열, dtype=int)` — `dtype=int`로 0/1 표시)

<출력결과>

Thur  Fri  Sat  Sun
0     0    0    0    1
1     0    0    0    1
2     0    0    0    1
3     0    0    0    1
4     0    0    0    1

In [ ]:
encoded = pd.get_dummies(tips['day'], dtype=int)
print(encoded.head())

문제 3) 여러 열 한 번에 + drop_first

- `sex`·`smoker`·`time` **세 범주 열을 한 번에** 원-핫 인코딩하되 `drop_first=True` 를 주세요. (힌트 : `pd.get_dummies(df, columns=[...], drop_first=True, dtype=int)`)
- 만들어진 **열 이름 목록**을 출력하고, 앞 5줄도 출력하세요.

<출력결과>

열: ['sex_Female', 'smoker_No', 'time_Dinner']
   sex_Female  smoker_No  time_Dinner
0           1          1            1
1           0          1            1
2           0          1            1
3           0          1            1
4           1          1            1

In [ ]:
encoded = pd.get_dummies(
    tips[['sex', 'smoker', 'time']],
    drop_first=True,
    dtype=int
)
target_cols = ['sex_Female', 'smoker_No', 'time_Dinner']

print(f"열 : {target_cols}")
print(encoded.head())

---

### [2부] EDA 미니 파이프라인

seaborn `mpg`(자동차 연비 데이터, 398대)로 **불러오기 → 결측치 처리 → 인코딩·스케일링 → 상관 분석** 의 흐름을 밟아봅니다. 아래 셀을 먼저 실행하세요.

- 수치 열 : `mpg`(연비), `cylinders`, `displacement`(배기량), `horsepower`(마력), `weight`(무게), `acceleration`
- 범주 열 : `origin`(제조국: usa/japan/europe)

In [ ]:
mpg = sns.load_dataset("mpg")
print(mpg.head(3))

In [ ]:
문제 4) 첫 탐색과 결측치 처리

- 열별 결측치 개수를 `isnull().sum()`으로 출력하세요. (`horsepower`에 빈칸이 있습니다)
- `horsepower`의 빈칸을 **중앙값**으로 채우고, 전체 결측치가 **0**이 됐는지 확인해 출력하세요.

<출력결과>

mpg             0
cylinders       0
displacement    0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
name            0
dtype: int64
남은 결측치: 0

In [ ]:
print(mpg.isnull().sum())
mpg['horsepower'] = mpg['horsepower'].fillna(mpg['horsepower'].median())
print(f"남은 결측치 : {mpg.isnull().sum().sum()}")

문제 5) 인코딩 + 스케일링

- `origin`(제조국)을 **원-핫 인코딩**해 앞 5줄 출력하세요. (`dtype=int`)
- `weight`·`horsepower` 두 열을 `StandardScaler`로 **표준화**하고, 결과를 소수 셋째 자리까지 앞 5줄 출력하세요.

<출력결과>

europe  japan  usa
0       0      0    1
1       0      0    1
2       0      0    1
3       0      0    1
4       0      0    1
   weight  horsepower
0   0.631       0.673
1   0.854       1.590
2   0.550       1.197
3   0.547       1.197
4   0.566       0.935

In [ ]:
from sklearn.preprocessing import StandardScaler
encoded = pd.get_dummies(
    mpg['origin'],
    dtype=int
)

scaled = pd.DataFrame(
    StandardScaler().fit_transform(mpg[['weight', 'horsepower']]),
    columns=['weight', 'horsepower']
)
print(encoded.head())
print(scaled.round(3).head())

문제 6) 상관 분석 — mpg와 관련 깊은 특성 찾기

- 수치 열 6개(`mpg`, `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration`)의 **상관행렬**을 소수 둘째 자리까지 출력하세요.
- `mpg`와 다른 열들의 상관계수만 뽑아 **값 크기순으로 정렬**해, **연비(mpg)와 가장 관련 깊은 특성**을 찾아보세요. (힌트 : `corr()["mpg"].drop("mpg").sort_values()`)

<출력결과>

mpg  cylinders  displacement  horsepower  weight  acceleration
mpg           1.00      -0.78         -0.80       -0.77   -0.83          0.42
cylinders    -0.78       1.00          0.95        0.84    0.90         -0.51
displacement -0.80       0.95          1.00        0.90    0.93         -0.54
horsepower   -0.77       0.84          0.90        1.00    0.86         -0.69
weight       -0.83       0.90          0.93        0.86    1.00         -0.42
acceleration  0.42      -0.51         -0.54       -0.69   -0.42          1.00

mpg
weight         -0.832
displacement   -0.804
cylinders      -0.775
horsepower     -0.773
acceleration    0.420
Name: mpg, dtype: float64

In [ ]:
cols = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
print(mpg[cols].corr().round(2))
print()
print(mpg[cols].corr()["mpg"].drop("mpg").sort_values())